[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/48_moe_advanced_solution.ipynb)

# Solution: MoE (Advanced: Top-k + Aux Loss)

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class AdvancedMoE(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model),
            )
            for _ in range(num_experts)
        ])

    def forward(self, x, return_aux_loss=False):
        bsz, seqlen, d_model = x.shape
        logits = self.router(x)

        x_flat = x.reshape(-1, d_model)
        logits_flat = logits.reshape(-1, self.num_experts)
        top_vals, top_idx = logits_flat.topk(self.top_k, dim=-1)
        top_w = torch.softmax(top_vals, dim=-1)

        out_flat = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            expert_ids = top_idx[:, k]
            weights = top_w[:, k]
            for e, expert in enumerate(self.experts):
                mask = expert_ids == e
                if mask.any():
                    out_flat[mask] += weights[mask].unsqueeze(-1) * expert(x_flat[mask])

        out = out_flat.reshape(bsz, seqlen, d_model)

        probs = torch.softmax(logits_flat, dim=-1)
        importance = probs.mean(dim=0)
        top1 = top_idx[:, 0]
        load = torch.bincount(top1, minlength=self.num_experts).to(dtype=probs.dtype) / top1.numel()
        aux_loss = self.num_experts * torch.sum(importance * load)

        if return_aux_loss:
            return out, aux_loss
        return out


In [ ]:
# Demo
moe = AdvancedMoE(d_model=32, d_ff=64, num_experts=4, top_k=2)
x = torch.randn(2, 6, 32)
out, aux = moe(x, return_aux_loss=True)
print('Output shape:', out.shape)
print('Aux loss:', float(aux))

In [ ]:
from torch_judge import check
check('moe_advanced')